[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/16_conditioning_and_dit.ipynb)

# 16. Conditioning and DiT modulation — from FiLM to MMDiT

이전 버전은 AdaLN-Zero를 branch 하나에 zero gate만 곱한 형태로 축소했고, MMDiT는 reference에 이름만 있었다.

이번 버전은 **FiLM → cross-attention → DiT adaLN-Zero 6-way modulation → MMDiT joint attention** 순서로 실제 구조 차이를 보여준다.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. FiLM: condition controls feature-wise scale and shift


In [ ]:
features = torch.randn(2, 5, 8, device=device)
condition = torch.randn(2, 4, device=device)

film_projection = nn.Linear(4, 16).to(device)
scale, shift = film_projection(condition).chunk(2, dim=-1)

film_output = (
    features * (1 + scale[:, None])
    + shift[:, None]
)

print("FiLM output:", film_output.shape)


## 2. Cross-attention: image/query tokens read condition tokens

latent diffusion 계열 cross-attention에서는 image/latent tokens에서 query를 만들고 text condition tokens에서 key/value를 만든다. 두 modality가 같은 sequence로 합쳐지는 MMDiT와는 다르다.


In [ ]:
image_tokens = torch.randn(2, 6, 12, device=device)
text_tokens = torch.randn(2, 4, 12, device=device)

query_projection = nn.Linear(12, 12, bias=False).to(device)
key_projection = nn.Linear(12, 12, bias=False).to(device)
value_projection = nn.Linear(12, 12, bias=False).to(device)

q = query_projection(image_tokens).view(2, 6, 3, 4).transpose(1, 2)
k = key_projection(text_tokens).view(2, 4, 3, 4).transpose(1, 2)
v = value_projection(text_tokens).view(2, 4, 3, 4).transpose(1, 2)

cross_attention = F.scaled_dot_product_attention(q, k, v)
print("cross-attention:", cross_attention.shape)


## 3. DiT adaLN-Zero: two modulated residual branches

DiT의 adaLN-Zero는 condition에서 단순 scale/shift 둘만 만드는 것이 아니다. attention branch와 MLP branch 각각에 `shift, scale, gate`를 만들어 **6개의 modulation vector**를 사용한다. modulation linear layer와 output layer를 zero-initialize해 network가 처음에는 identity에 가깝게 시작한다.


In [ ]:
def modulate(x, shift, scale):
    return x * (1 + scale[:, None]) + shift[:, None]


class TinyAdaLNZeroBlock(nn.Module):
    def __init__(self, hidden_dim=12, num_heads=3):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            hidden_dim,
            elementwise_affine=False,
        )
        self.attention = nn.MultiheadAttention(
            hidden_dim,
            num_heads,
            batch_first=True,
        )

        self.norm2 = nn.LayerNorm(
            hidden_dim,
            elementwise_affine=False,
        )
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * hidden_dim, hidden_dim),
        )

        self.modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_dim, 6 * hidden_dim),
        )
        nn.init.zeros_(self.modulation[-1].weight)
        nn.init.zeros_(self.modulation[-1].bias)

    def forward(self, x, condition):
        (
            shift_attn,
            scale_attn,
            gate_attn,
            shift_mlp,
            scale_mlp,
            gate_mlp,
        ) = self.modulation(condition).chunk(6, dim=-1)

        attention_input = modulate(
            self.norm1(x),
            shift_attn,
            scale_attn,
        )
        attention_output, _ = self.attention(
            attention_input,
            attention_input,
            attention_input,
            need_weights=False,
        )
        x = x + gate_attn[:, None] * attention_output

        mlp_input = modulate(
            self.norm2(x),
            shift_mlp,
            scale_mlp,
        )
        x = x + gate_mlp[:, None] * self.mlp(mlp_input)

        return x


x = torch.randn(2, 6, 12, device=device)
condition_vector = torch.randn(2, 12, device=device)
block = TinyAdaLNZeroBlock().to(device)

output = block(x, condition_vector)
print("max initial residual change:", (output - x).abs().max().item())


## 4. MMDiT: separate modality projections, joint attention

Stable Diffusion 3의 MMDiT 계열은 text와 image tokens를 단순 cross-attention으로 한쪽에서 읽는 방식과 다르다. 각 modality가 **자기 LayerNorm/QKV projection을 따로 가지고**, 만들어진 Q/K/V를 sequence dimension으로 합쳐 **joint attention**을 수행한 뒤 다시 image/text branch로 나눈다. branch별 MLP parameter도 따로 둘 수 있다.


In [ ]:
class ModalityAttentionProjection(nn.Module):
    def __init__(self, hidden_dim=12, num_heads=3):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.norm = nn.LayerNorm(hidden_dim)
        self.qkv = nn.Linear(hidden_dim, 3 * hidden_dim)
        self.out = nn.Linear(hidden_dim, hidden_dim)

    def project_qkv(self, x):
        batch_size, sequence_length, hidden_dim = x.shape
        qkv = self.qkv(self.norm(x))
        qkv = qkv.view(
            batch_size,
            sequence_length,
            3,
            self.num_heads,
            self.head_dim,
        ).permute(2, 0, 3, 1, 4)
        return qkv.unbind(0)


class TinyMMDiTBlock(nn.Module):
    def __init__(self, hidden_dim=12, num_heads=3):
        super().__init__()

        self.image_attention = ModalityAttentionProjection(
            hidden_dim,
            num_heads,
        )
        self.text_attention = ModalityAttentionProjection(
            hidden_dim,
            num_heads,
        )

        self.image_norm2 = nn.LayerNorm(hidden_dim)
        self.text_norm2 = nn.LayerNorm(hidden_dim)

        self.image_mlp = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim),
            nn.GELU(),
            nn.Linear(4 * hidden_dim, hidden_dim),
        )
        self.text_mlp = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim),
            nn.GELU(),
            nn.Linear(4 * hidden_dim, hidden_dim),
        )

    def forward(self, image_tokens, text_tokens):
        image_q, image_k, image_v = self.image_attention.project_qkv(
            image_tokens
        )
        text_q, text_k, text_v = self.text_attention.project_qkv(
            text_tokens
        )

        joint_q = torch.cat([image_q, text_q], dim=2)
        joint_k = torch.cat([image_k, text_k], dim=2)
        joint_v = torch.cat([image_v, text_v], dim=2)

        joint_output = F.scaled_dot_product_attention(
            joint_q,
            joint_k,
            joint_v,
        )

        image_length = image_tokens.size(1)
        image_output = joint_output[:, :, :image_length]
        text_output = joint_output[:, :, image_length:]

        def merge_heads(x):
            return x.transpose(1, 2).contiguous().flatten(2)

        image_tokens = image_tokens + self.image_attention.out(
            merge_heads(image_output)
        )
        text_tokens = text_tokens + self.text_attention.out(
            merge_heads(text_output)
        )

        image_tokens = image_tokens + self.image_mlp(
            self.image_norm2(image_tokens)
        )
        text_tokens = text_tokens + self.text_mlp(
            self.text_norm2(text_tokens)
        )

        return image_tokens, text_tokens


image_tokens = torch.randn(2, 6, 12, device=device)
text_tokens = torch.randn(2, 4, 12, device=device)
mmdit = TinyMMDiTBlock().to(device)

image_output, text_output = mmdit(
    image_tokens,
    text_tokens,
)

print("image output:", image_output.shape)
print("text output:", text_output.shape)
print("joint attention token count:", 6 + 4)


## References and provenance

**FiLM** — Perez et al. feature-wise affine conditioning을 반영했다.

**Cross-attention** — Transformer encoder-decoder / latent diffusion 계열. query source와 key/value condition source가 분리되는 구조를 반영했다.

**DiT** — Peebles & Xie 및 공식 DiT 구현. attention/MLP 각각의 shift-scale-gate, 6-way adaLN modulation, zero initialization을 반영했다.

**MMDiT / Stable Diffusion 3** — Esser et al., *Scaling Rectified Flow Transformers for High-Resolution Image Synthesis*. modality-specific projections/parameters와 concatenated joint attention의 핵심을 작은 형태로 반영했다. FLUX/후속 multimodal DiT는 세부 block 설계가 다르므로 같은 구조라고 취급하지 않는다.
